In [1]:
"""
08 - Robustness Check: Regressions with Control Variables
---------------------------------------------------------
Re-estimates the cross-sectional regression in 06,
adding one control variable at a time.

DV       : CAR
IV       : LEAD, TECH, RD, BM, TECHxRD
Controls : SIZE - ln(total assets)
           LEV  - total debt / total assets
           (both from the fiscal year preceding the announcement)

Data source : firm_fund.xlsx
"""

import pandas as pd
import statsmodels.api as sm

df = pd.read_excel("../data/firm_fund.xlsx")

# Centering for interaction (same as 06)
df["RD_c"] = df["RD"] - df["RD"].mean()
df["TECH_RD"] = df["TECH"] * df["RD_c"]

windows = {
    "(-1, +1)": "CAR -1~+1",
    "(-5, +5)": "CAR -5~+5",
    "(-10, +10)": "CAR -10~+10",
}

controls = {
    "Panel A: Controlling for Firm Size": "SIZE",
    "Panel B: Controlling for Financial Leverage": "LEV",
}

def star(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""

for panel, ctrl in controls.items():
    X = sm.add_constant(df[[ctrl, "LEAD", "TECH", "RD", "BM", "TECH_RD"]])
    table = {}
    for name, col in windows.items():
        model = sm.OLS(df[col], X).fit()
        cells = [
            f"{model.params[v]:.2f}{star(model.pvalues[v])} ({model.tvalues[v]:.2f})"
            for v in X.columns
        ]
        cells += [f"{int(model.nobs)}",
                  f"{model.rsquared:.1%}",
                  f"{model.rsquared_adj:.1%}"]
        table[name] = cells

    index = ["Intercept"] + list(X.columns[1:]) + ["Observations", "R2", "Adjusted R2"]
    out = pd.DataFrame(table, index=index)
    out.index = out.index.str.replace("TECH_RD", "TECH x RD")

    print(panel)
    print(out.to_string())
    print("\n* p < 0.10, ** p < 0.05, *** p < 0.01. t-statistics in parentheses.\n")

Panel A: Controlling for Firm Size
                    (-1, +1)        (-5, +5)         (-10, +10)
Intercept      -4.30 (-0.27)  -21.24 (-0.92)      -1.27 (-0.06)
SIZE             0.42 (0.45)     1.15 (0.83)        0.15 (0.13)
LEAD             0.00 (0.00)     4.13 (0.75)       9.71* (2.02)
TECH           -0.62 (-0.16)    10.20 (1.77)     13.45** (2.66)
RD              42.75 (0.88)    34.36 (0.49)       33.68 (0.55)
BM             -4.61 (-0.87)  -13.35 (-1.75)  -24.38*** (-3.65)
TECH x RD     -45.35 (-0.92)  -83.79 (-1.17)   -141.07* (-2.26)
Observations              15              15                 15
R2                     18.7%           58.5%              80.4%
Adjusted R2           -42.3%           27.4%              65.6%

* p < 0.10, ** p < 0.05, *** p < 0.01. t-statistics in parentheses.

Panel B: Controlling for Financial Leverage
                    (-1, +1)        (-5, +5)          (-10, +10)
Intercept        3.91 (0.40)   -7.33 (-0.51)     -17.76* (-1.89)
LEV            -1